# NB-03 · Domain-Specific Model Architecture Ideation
## Clinical Decision Making · Revenue Cycle · Patient Engagement · Knowledge Graph


Architecture blueprints with prototype implementations for three distinct
healthcare AI use-cases, unified by a shared ICD-10 knowledge graph.

| Section | Architecture |
|---------|-------------|
| 1 | Setup |
| 2 | Clinical TFT — Clinical Decision Support |
| 3 | HDE-CGD — Revenue Cycle Management |
| 4 | MOCB-LLM — Patient Engagement |
| 5 | Unified ICD-10 Knowledge Graph |
| 6 | KG Queries & Visualisation |

In [1]:
import subprocess, sys

def uv_install(*packages: str) -> None:
    """Install via uv, fallback to pip."""
    try:
        subprocess.run([sys.executable, '-m', 'uv', 'pip', 'install', '--quiet', *packages], check=True)
    except (subprocess.CalledProcessError, FileNotFoundError):
        try:
            subprocess.run([sys.executable, '-m', 'pip', 'install', '--quiet', *packages], check=True)
        except subprocess.CalledProcessError:
            print(f'⚠️  Some packages may have failed to install, continuing with existing environment')

uv_install(
    'torch>=2.2', 'numpy>=1.26', 'networkx>=3.3',
    'pyvis>=0.3', 'plotly>=5.20', 'scikit-learn>=1.4',
    'pandas>=2.2',
)
print('✅  Dependencies ready')

✅  Dependencies ready


In [2]:
from __future__ import annotations
import logging, random
import numpy as np
import torch
import torch.nn as nn

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
logging.basicConfig(level=logging.INFO, format='%(asctime)s | %(levelname)-8s | %(message)s', datefmt='%H:%M:%S')
log = logging.getLogger('cliniq.nb03')
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')

## § 2 · Clinical Temporal Fusion Transformer (C-TFT)

**Use-case:** Given longitudinal patient history (encounters, labs, meds) → differential diagnosis suggestion + risk score.

**Key innovation:** Gating mechanism fuses static patient demographics with
temporal event embeddings. Knowledge graph neighbourhood embedding injects
structural ICD-10 prior directly into the model.

### Why Gating for Static × Temporal Fusion?

A naive approach would **concatenate** static features with each temporal timestep. But age and comorbidity count are *constant* across the whole sequence — concatenating them adds noise to each attention computation.

The **gating mechanism** learns *when* to let static context modulate temporal attention:
$$g = \sigma(W_g [h_{temporal}, c_{static}])$$
$$h_{out} = \text{LayerNorm}(h_{temporal} + g \cdot \text{Attn}(h_{temporal}))$$

The gate $g \in [0,1]$ can learn: 'for an elderly patient ($c_{static}$ = high age), amplify attention to recent lab deterioration (temporal)'. This is the intuition behind Temporal Fusion Transformer (Lim et al., 2021).

**KG embedding role:** ICD-10 node2vec embeddings encode structural relationships — 'N18.3 is close to E11.9 in the graph' means 'patients with CKD stage 3 often also have T2DM'. Injecting this prior reduces the labelled data needed for rare diagnosis combinations.

In [3]:
class StaticCovariateEncoder(nn.Module):
    """Encodes static patient features (age, gender, comorbidity count) into context vector."""

    def __init__(self, in_dim: int, out_dim: int) -> None:
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(in_dim, out_dim * 2), nn.SiLU(),
            nn.Linear(out_dim * 2, out_dim),
        )

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        return self.net(x)


class TemporalGatedAttention(nn.Module):
    """Multi-head self-attention gated by static context vector."""

    def __init__(self, d_model: int, n_heads: int, dropout: float = 0.1) -> None:
        super().__init__()
        self.attn = nn.MultiheadAttention(d_model, n_heads, dropout=dropout, batch_first=True)
        self.gate = nn.Linear(d_model * 2, d_model)
        self.norm = nn.LayerNorm(d_model)

    def forward(
        self, temporal: torch.Tensor, static_ctx: torch.Tensor
    ) -> torch.Tensor:
        attn_out, _ = self.attn(temporal, temporal, temporal)
        # Static context modulates how much attention output is passed through
        ctx_expanded = static_ctx.unsqueeze(1).expand_as(attn_out)
        gate_val = torch.sigmoid(self.gate(torch.cat([attn_out, ctx_expanded], dim=-1)))
        return self.norm(temporal + gate_val * attn_out)


class ClinicalTFT(nn.Module):
    """Clinical Temporal Fusion Transformer for longitudinal patient modelling.
    
    Architecture:
        Static Covariate Encoder → context vector
        Temporal Event Embedder (per-timestep) → sequence
        Gated Multi-head Attention (temporal × static context)
        KG Neighbourhood Embedding concatenation
        Multi-task head: diagnosis classification + risk score
    """

    def __init__(
        self,
        n_static:  int,
        n_temporal: int,
        d_model:   int = 64,
        n_heads:   int = 4,
        n_layers:  int = 2,
        n_classes: int = 10,
        kg_dim:    int = 16,     # KG neighbourhood embedding dimension
        dropout:   float = 0.1,
    ) -> None:
        super().__init__()
        self.static_enc   = StaticCovariateEncoder(n_static, d_model)
        self.temporal_proj = nn.Linear(n_temporal, d_model)
        self.gated_attn   = nn.ModuleList([
            TemporalGatedAttention(d_model, n_heads, dropout)
            for _ in range(n_layers)
        ])
        self.kg_proj      = nn.Linear(kg_dim, d_model // 4)
        fusion_dim        = d_model + d_model // 4
        self.diag_head    = nn.Linear(fusion_dim, n_classes)   # diagnosis
        self.risk_head    = nn.Linear(fusion_dim, 1)            # readmission risk
        self.drop         = nn.Dropout(dropout)

    def forward(
        self,
        static:   torch.Tensor,   # (B, n_static)
        temporal: torch.Tensor,   # (B, T, n_temporal)
        kg_emb:   torch.Tensor,   # (B, kg_dim)
    ) -> dict[str, torch.Tensor]:
        ctx  = self.static_enc(static)                     # (B, d_model)
        h    = self.temporal_proj(temporal)                # (B, T, d_model)
        for layer in self.gated_attn:
            h = layer(h, ctx)
        h_pool  = h.mean(dim=1)                           # mean pool over time
        kg_feat = self.kg_proj(kg_emb)                    # (B, d_model//4)
        fused   = self.drop(torch.cat([h_pool, kg_feat], dim=-1))
        return {
            'diagnosis_logits': self.diag_head(fused),
            'risk_score':       torch.sigmoid(self.risk_head(fused)).squeeze(-1),
        }


# ── Smoke test ────────────────────────────────────────────────────────
model_cds = ClinicalTFT(n_static=5, n_temporal=8, d_model=64, n_classes=10, kg_dim=16)
B, T = 4, 12
out = model_cds(
    static  =torch.randn(B, 5),
    temporal=torch.randn(B, T, 8),
    kg_emb  =torch.randn(B, 16),
)
log.info('C-TFT | diag logits: %s | risk: %s', out['diagnosis_logits'].shape, out['risk_score'].shape)
print(f'✅  C-TFT forward pass OK — params: {sum(p.numel() for p in model_cds.parameters()):,}')

23:28:59 | INFO     | C-TFT | diag logits: torch.Size([4, 10]) | risk: torch.Size([4])


✅  C-TFT forward pass OK — params: 60,811


## § 3 · Hierarchical Document Encoder + Code Graph Decoder (HDE-CGD)

**Use-case:** Clinical documentation → generate ICD-10 code set (RCM automation).

**Key innovation:** Decoder is constrained to ICD-10 code graph — beam search
can only emit codes that are valid children of the current ICD-10 node,
eliminating impossible code combinations without any post-processing.

In [4]:
class HierarchicalDocumentEncoder(nn.Module):
    """Two-level encoder: sentence → document using separate transformer layers."""

    def __init__(
        self, vocab_size: int, d_model: int = 64,
        n_heads: int = 4, dropout: float = 0.1
    ) -> None:
        super().__init__()
        self.word_embed  = nn.Embedding(vocab_size, d_model, padding_idx=0)
        # Sentence-level encoder
        sent_layer  = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4,
                                                 dropout, batch_first=True, norm_first=True)
        self.sent_enc = nn.TransformerEncoder(sent_layer, num_layers=2)
        # Document-level encoder (over sentence representations)
        doc_layer   = nn.TransformerEncoderLayer(d_model, n_heads, d_model * 4,
                                                 dropout, batch_first=True, norm_first=True)
        self.doc_enc  = nn.TransformerEncoder(doc_layer, num_layers=2)

    def forward(self, sents: torch.Tensor) -> torch.Tensor:
        # sents: (B, n_sents, sent_len)
        B, S, L = sents.shape
        flat = sents.view(B * S, L)                    # (B*S, L)
        sent_repr = self.sent_enc(self.word_embed(flat))[:,0]  # CLS per sentence
        sent_repr = sent_repr.view(B, S, -1)           # (B, S, D)
        doc_repr  = self.doc_enc(sent_repr)            # (B, S, D)
        return doc_repr.mean(dim=1)                    # (B, D)


class GraphConstrainedCodeDecoder(nn.Module):
    """Greedy decoder that respects ICD-10 hierarchy constraints.
    
    In production: replace greedy with constrained beam search.
    Here we demonstrate the constraint logic with a toy vocabulary.
    """

    def __init__(
        self, code_vocab_size: int, d_model: int = 64,
        max_codes: int = 5, dropout: float = 0.1
    ) -> None:
        super().__init__()
        self.code_embed = nn.Embedding(code_vocab_size, d_model)
        self.attn       = nn.MultiheadAttention(d_model, 4, dropout=dropout, batch_first=True)
        self.head       = nn.Linear(d_model, code_vocab_size)
        self.max_codes  = max_codes

    def forward(
        self,
        doc_repr: torch.Tensor,       # (B, D) from encoder
        code_mask: torch.Tensor,      # (B, code_vocab_size) — 0=allowed, -inf=forbidden
    ) -> torch.Tensor:
        # Expand doc_repr as query sequence of length 1
        q = doc_repr.unsqueeze(1)                     # (B, 1, D)
        # Attend over code embeddings as keys/values
        codes = self.code_embed.weight.unsqueeze(0).expand(q.size(0), -1, -1)
        out, _ = self.attn(q, codes, codes)
        logits = self.head(out.squeeze(1))            # (B, code_vocab_size)
        # Apply hierarchy mask
        return logits + code_mask


# ── Smoke test ────────────────────────────────────────────────────────
VOCAB_SIZE_SMALL = 100
CODE_VOCAB       = 50
enc = HierarchicalDocumentEncoder(VOCAB_SIZE_SMALL, d_model=64)
dec = GraphConstrainedCodeDecoder(CODE_VOCAB, d_model=64)

B, S, L = 2, 5, 15
doc_out  = enc(torch.randint(0, VOCAB_SIZE_SMALL, (B, S, L)))
mask     = torch.zeros(B, CODE_VOCAB)  # all codes allowed in this example
logits   = dec(doc_out, mask)
log.info('HDE-CGD | doc_repr: %s | logits: %s', doc_out.shape, logits.shape)
print(f'✅  HDE-CGD forward pass OK')

C:\Users\prave\AppData\Roaming\Python\Python314\site-packages\torch\nn\modules\transformer.py:392: UserWarning: enable_nested_tensor is True, but self.use_nested_tensor is False because encoder_layer.norm_first was True
  warnings.warn(
23:28:59 | INFO     | HDE-CGD | doc_repr: torch.Size([2, 64]) | logits: torch.Size([2, 50])


✅  HDE-CGD forward pass OK


In [5]:
# ── Demonstrate ICD-10 hierarchy constraint on decoder ──────────────
# Hierarchy rule: if parent N18 is selected, only N18.x children are valid next codes

# Build a toy ICD-10 vocab for the demo
ICD10_DEMO_VOCAB = ['<PAD>', 'N18', 'N18.1', 'N18.2', 'N18.3',
                    'E11', 'E11.9', 'I10', 'I50', 'I50.9', '<EOS>']
CHILDREN = {'N18': ['N18.1','N18.2','N18.3'], 'E11': ['E11.9'], 'I50': ['I50.9']}

def build_hierarchy_mask(current_code: str, vocab: list[str]) -> torch.Tensor:
    """Return -inf mask for codes NOT allowed after current_code."""
    allowed = set(CHILDREN.get(current_code, vocab))   # if no children → any code allowed
    allowed.add('<EOS>')
    mask = torch.full((len(vocab),), float('-inf'))
    for i, c in enumerate(vocab):
        if c in allowed:
            mask[i] = 0.0
    return mask

print('\n📊 Graph-Constrained Decoding Demo')
print('─' * 45)
for selected_parent in ['N18', 'E11', 'I10']:
    mask   = build_hierarchy_mask(selected_parent, ICD10_DEMO_VOCAB)
    allowed = [c for c, m in zip(ICD10_DEMO_VOCAB, mask.tolist()) if m == 0.0]
    print(f'  After selecting [{selected_parent}] → valid next codes: {allowed}')

print('\n💡 This constraint eliminates invalid code sequences without post-processing.')
print('   In production: beam search is constrained at each decoding step.')


📊 Graph-Constrained Decoding Demo
─────────────────────────────────────────────
  After selecting [N18] → valid next codes: ['N18.1', 'N18.2', 'N18.3', '<EOS>']
  After selecting [E11] → valid next codes: ['E11.9', '<EOS>']
  After selecting [I10] → valid next codes: ['<PAD>', 'N18', 'N18.1', 'N18.2', 'N18.3', 'E11', 'E11.9', 'I10', 'I50', 'I50.9', '<EOS>']

💡 This constraint eliminates invalid code sequences without post-processing.
   In production: beam search is constrained at each decoding step.


## § 4 · Multi-objective Contextual Bandit + LLM Generator (MOCB-LLM)

**Use-case:** Patient engagement — personalise outreach channel, timing, and message content.

**Key innovation:** LinUCB bandit selects optimal (channel, timing, tone) action;
Qwen2.5 generates the actual message conditioned on the bandit action.
Fairness constraint ensures equal engagement opportunity across demographic groups.

### LinUCB: The Mathematics of Contextual Bandits

LinUCB assumes the expected reward is **linear in context**: $\mathbb{E}[r|x, a] = \theta_a^T x$.

**Parameter update** (ridge regression on observed rewards):
$$A_a \leftarrow A_a + x_t x_t^T \qquad b_a \leftarrow b_a + r_t x_t$$
$$\hat{\theta}_a = A_a^{-1} b_a$$

**Action selection** (UCB — upper confidence bound):
$$a^* = \arg\max_a \left[ \hat{\theta}_a^T x_t + \alpha \sqrt{x_t^T A_a^{-1} x_t} \right]$$

The term $\sqrt{x_t^T A_a^{-1} x_t}$ is the **uncertainty** (standard deviation of the reward estimate). High uncertainty → explore this action. $\alpha$ controls the exploration-exploitation tradeoff.

**Why this works for patient engagement:**
- Context $x_t$ = patient features (age, insurance, lab values, prior engagement)
- Actions = (channel, timing, tone) combinations
- Reward = whether patient acted (opened app, kept appointment, took medication)
- The model learns *per-patient* optimal outreach strategy from historical outcomes

In [6]:
import numpy as np
from dataclasses import dataclass, field
from typing import Literal

@dataclass
class BanditAction:
    """Represents one outreach action chosen by the bandit."""
    channel: Literal['sms', 'email', 'app_push', 'phone']
    timing:  Literal['morning', 'afternoon', 'evening']
    tone:    Literal['empathetic', 'informational', 'motivational']

    def to_id(self) -> int:
        channels = ['sms', 'email', 'app_push', 'phone']
        timings  = ['morning', 'afternoon', 'evening']
        tones    = ['empathetic', 'informational', 'motivational']
        return (channels.index(self.channel) * 9
                + timings.index(self.timing) * 3
                + tones.index(self.tone))

ALL_ACTIONS = [
    BanditAction(ch, ti, to)
    for ch in ['sms', 'email', 'app_push', 'phone']
    for ti in ['morning', 'afternoon', 'evening']
    for to in ['empathetic', 'informational', 'motivational']
]
N_ACTIONS = len(ALL_ACTIONS)


class LinUCBBandit:
    """LinUCB contextual bandit with fairness constraint.
    
    Fairness: if cumulative selection rate for any demographic group
    falls below min_equity_ratio, boost UCB score for under-served groups.
    """

    def __init__(
        self,
        n_actions:       int,
        context_dim:     int,
        alpha:           float = 1.0,
        min_equity_ratio: float = 0.8,
    ) -> None:
        self.alpha           = alpha
        self.min_equity_ratio = min_equity_ratio
        self.n_actions       = n_actions
        self.context_dim     = context_dim
        self.A = [np.eye(context_dim) for _ in range(n_actions)]
        self.b = [np.zeros((context_dim, 1)) for _ in range(n_actions)]
        self.group_counts: dict[str, int] = {}  # group → times selected

    def select_action(
        self,
        context: np.ndarray,
        patient_group: str,
        patient_groups: list[str],
    ) -> int:
        """UCB action selection with fairness boost for under-served groups."""
        x = context.reshape(-1, 1)  # (d, 1)
        p_values = np.zeros(self.n_actions)
        for a in range(self.n_actions):
            A_inv = np.linalg.inv(self.A[a])
            theta = A_inv @ self.b[a]
            p_values[a] = (theta.T @ x).item() + self.alpha * np.sqrt((x.T @ A_inv @ x).item())

        # Fairness: compute equity ratio for this patient's group
        total_counts = sum(self.group_counts.values()) + 1
        group_count  = self.group_counts.get(patient_group, 0)
        n_groups     = len(set(patient_groups))
        equity_ratio = (group_count / total_counts) / (1 / n_groups + 1e-9)
        if equity_ratio < self.min_equity_ratio:
            p_values += 0.5   # boost all actions for under-served group

        chosen = int(np.argmax(p_values))
        self.group_counts[patient_group] = group_count + 1
        return chosen

    def update(self, action: int, context: np.ndarray, reward: float) -> None:
        """Update model parameters for selected action."""
        x = context.reshape(-1, 1)
        self.A[action] += x @ x.T
        self.b[action] += reward * x


# ── Simulation ────────────────────────────────────────────────────────
CONTEXT_DIM = 8
N_SIM       = 200
GROUPS      = ['senior', 'mid_age', 'young', 'medicaid']
bandit      = LinUCBBandit(N_ACTIONS, CONTEXT_DIM, alpha=1.0, min_equity_ratio=0.8)

cumulative_reward = 0.0
rewards_log: list[float] = []

for step in range(N_SIM):
    ctx   = np.random.randn(CONTEXT_DIM)
    group = random.choice(GROUPS)
    action_idx = bandit.select_action(ctx, group, GROUPS)
    action = ALL_ACTIONS[action_idx]
    # Simulate reward: channel-timing interaction (synthetic ground truth)
    true_best = 'sms' if group == 'medicaid' else 'email'
    reward = 1.0 if action.channel == true_best else 0.0
    bandit.update(action_idx, ctx, reward)
    cumulative_reward += reward
    rewards_log.append(cumulative_reward / (step + 1))

log.info('Bandit simulation done. Final avg reward: %.3f', rewards_log[-1])
print(f'\n✅  LinUCB Bandit simulation: avg reward = {rewards_log[-1]:.3f}')
print(f'   Group engagement counts: {bandit.group_counts}')

23:29:00 | INFO     | Bandit simulation done. Final avg reward: 0.350



✅  LinUCB Bandit simulation: avg reward = 0.350
   Group engagement counts: {'senior': 56, 'young': 43, 'mid_age': 57, 'medicaid': 44}


In [7]:
# ── LinUCB regret curve ────────────────────────────────────────────────
import plotly.graph_objects as go

# Compute oracle reward rate (always picks best channel per group)
oracle_rate = 0.50   # theoretical best: 50% correct given 2 channels (sms for medicaid, email for rest)
regret = [oracle_rate - r for r in rewards_log]

fig_regret = go.Figure()
fig_regret.add_trace(go.Scatter(
    y=rewards_log, name='LinUCB avg reward',
    line=dict(color='#0E7C7B'), mode='lines',
))
fig_regret.add_hline(y=oracle_rate, line_dash='dash', line_color='#DC2626',
                      annotation_text='Oracle (optimal)')
fig_regret.update_layout(
    title='LinUCB Learning Curve — Patient Engagement Bandit',
    xaxis_title='Step', yaxis_title='Average Reward',
    template='plotly_white',
)
fig_regret.show()

# Group equity analysis
print('\n📊 Group Engagement Equity')
total_steps = sum(bandit.group_counts.values())
for grp, cnt in bandit.group_counts.items():
    ratio = cnt / total_steps
    expected = 1 / len(set(bandit.group_counts.keys()))
    flag = '⚠️' if abs(ratio - expected) > 0.1 else '✅'
    print(f'  {flag} {grp:<12} {cnt:3d} steps ({ratio:.2%}) — expected {expected:.2%}')


📊 Group Engagement Equity
  ✅ senior        56 steps (28.00%) — expected 25.00%
  ✅ young         43 steps (21.50%) — expected 25.00%
  ✅ mid_age       57 steps (28.50%) — expected 25.00%
  ✅ medicaid      44 steps (22.00%) — expected 25.00%


## § 5 · Unified ICD-10 Knowledge Graph

In [8]:
import networkx as nx
from IPython.display import display

# ── Build ICD-10 subgraph (15 codes + 3 qualifier nodes = 18 nodes, 3 domains) ─
ICD10_NODES = {
    # Renal
    'N18':   {'desc': 'Chronic kidney disease (CKD)', 'chapter': 'N', 'domain': 'clinical'},
    'N18.1': {'desc': 'CKD, stage 1',                 'chapter': 'N', 'domain': 'clinical'},
    'N18.2': {'desc': 'CKD, stage 2',                 'chapter': 'N', 'domain': 'clinical'},
    'N18.3': {'desc': 'CKD, stage 3',                 'chapter': 'N', 'domain': 'clinical'},
    'N18.4': {'desc': 'CKD, stage 4',                 'chapter': 'N', 'domain': 'clinical'},
    'N18.5': {'desc': 'CKD, stage 5',                 'chapter': 'N', 'domain': 'rcm'},
    'N18.6': {'desc': 'End stage renal disease',      'chapter': 'N', 'domain': 'rcm'},
    # Endocrine
    'E11':   {'desc': 'Type 2 diabetes mellitus',     'chapter': 'E', 'domain': 'clinical'},
    'E11.9': {'desc': 'T2DM without complications',   'chapter': 'E', 'domain': 'clinical'},
    'E11.65':{'desc': 'T2DM with hyperglycaemia',     'chapter': 'E', 'domain': 'rcm'},
    'E11.40':{'desc': 'T2DM with diabetic neuropathy','chapter': 'E', 'domain': 'clinical'},
    # Cardiovascular
    'I10':   {'desc': 'Essential hypertension',       'chapter': 'I', 'domain': 'engagement'},
    'I50':   {'desc': 'Heart failure',                'chapter': 'I', 'domain': 'engagement'},
    'I50.9': {'desc': 'Heart failure, unspecified',   'chapter': 'I', 'domain': 'rcm'},
    'I50.32':{'desc': 'Chronic systolic HF',          'chapter': 'I', 'domain': 'rcm'},
}

HIERARCHY_EDGES = [
    ('N18', 'N18.1'), ('N18', 'N18.2'), ('N18', 'N18.3'),
    ('N18', 'N18.4'), ('N18', 'N18.5'), ('N18', 'N18.6'),
    ('E11', 'E11.9'), ('E11', 'E11.65'), ('E11', 'E11.40'),
    ('I50', 'I50.9'), ('I50', 'I50.32'),
]
CO_CODED_EDGES = [
    ('N18.3', 'E11.9'),  # CKD + T2DM very commonly co-coded
    ('N18.3', 'I10'),
    ('E11.9', 'I10'),
    ('I50.32', 'I10'),
    ('N18.6', 'E11.65'),
]
REQUIRES_EDGES = [
    ('N18', 'stage_qualifier'),
    ('E11', 'complication_qualifier'),
    ('I50', 'type_qualifier'),
]

G = nx.DiGraph()
for code, attrs in ICD10_NODES.items():
    G.add_node(code, **attrs)
for src, tgt in HIERARCHY_EDGES:
    G.add_edge(src, tgt, rel='PARENT_OF')
for src, tgt in CO_CODED_EDGES:
    G.add_edge(src, tgt, rel='COMMONLY_CO_CODED')
# Add qualifier nodes
for qualifier in ['stage_qualifier', 'complication_qualifier', 'type_qualifier']:
    G.add_node(qualifier, desc=qualifier, domain='qualifier', chapter='Q')
for src, tgt in REQUIRES_EDGES:
    G.add_edge(src, tgt, rel='REQUIRES')

log.info('KG built: %d nodes, %d edges', G.number_of_nodes(), G.number_of_edges())
print(f'\n✅  Knowledge Graph: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')

23:29:03 | INFO     | KG built: 18 nodes, 19 edges



✅  Knowledge Graph: 18 nodes, 19 edges


## § 6 · KG Queries & Visualisation

In [9]:
# ── Useful KG query functions ──────────────────────────────────────────
def get_commonly_co_coded(G: nx.DiGraph, code: str, top_k: int = 5) -> list[str]:
    """Return top-k commonly co-coded diagnoses for a given ICD-10 code."""
    neighbours = [
        tgt for src, tgt, data in G.edges(code, data=True)
        if data.get('rel') == 'COMMONLY_CO_CODED'
    ]
    return neighbours[:top_k]

def get_required_qualifiers(G: nx.DiGraph, code: str) -> list[str]:
    """Return qualifier types required for correct code specificity."""
    # Check code itself and its parents
    required = []
    try:
        ancestors = list(nx.ancestors(G, code))
    except nx.NetworkXError:
        ancestors = []
    check_codes = [code] + ancestors
    for c in check_codes:
        for src, tgt, data in G.edges(c, data=True):
            if data.get('rel') == 'REQUIRES':
                required.append(tgt)
    return list(set(required))

def get_children(G: nx.DiGraph, code: str) -> list[str]:
    """Return all direct child codes (more specific subcodes)."""
    return [tgt for src, tgt, data in G.edges(code, data=True)
            if data.get('rel') == 'PARENT_OF']

# Demo queries
print('── KG Queries ─────────────────────────────────────────────────')
print(f'Co-coded with N18.3: {get_commonly_co_coded(G, "N18.3")}')
print(f'Required qualifiers for N18: {get_required_qualifiers(G, "N18")}')
print(f'Children of E11: {get_children(G, "E11")}')
print(f'Centraliy (degree): {dict(sorted(nx.degree_centrality(G).items(), key=lambda x: -x[1])[:5])}')

── KG Queries ─────────────────────────────────────────────────
Co-coded with N18.3: ['E11.9', 'I10']
Required qualifiers for N18: ['stage_qualifier']
Children of E11: ['E11.9', 'E11.65', 'E11.40']
Centraliy (degree): {'N18': 0.4117647058823529, 'E11': 0.23529411764705882, 'N18.3': 0.1764705882352941, 'E11.9': 0.1764705882352941, 'I10': 0.1764705882352941}


In [10]:
# ── KG centrality interpretation — clinical meaning ──────────────────
import networkx as nx

centrality = nx.degree_centrality(G)
betweenness = nx.betweenness_centrality(G, normalized=True)

print('\n📊 Knowledge Graph Clinical Interpretation')
print('─' * 55)
print('\n  Degree Centrality (most connected = most commonly involved):')
for node, score in sorted(centrality.items(), key=lambda x: -x[1])[:5]:
    desc = G.nodes[node].get('desc', node)
    print(f'  {node:<10} {score:.3f}  — {desc[:45]}')

print()
print('  Betweenness Centrality (bridges between clinical domains):')
for node, score in sorted(betweenness.items(), key=lambda x: -x[1])[:5]:
    desc = G.nodes[node].get('desc', node)
    print(f'  {node:<10} {score:.3f}  — {desc[:45]}')

print()
print('💡 High degree centrality = commonly co-coded diagnosis = high CDI query priority.')
print('   I10 (hypertension) being central means: when you see hypertension, always')
print('   check for co-coded CKD, diabetes, and heart failure — they drive DRG weight.')
print()
print('   Betweenness centrality = diagnostic bridges: N18.3 connects renal, endocrine,')
print('   and cardiovascular clusters. Missing it means missing 3 revenue streams.')



📊 Knowledge Graph Clinical Interpretation
───────────────────────────────────────────────────────

  Degree Centrality (most connected = most commonly involved):
  N18        0.412  — Chronic kidney disease (CKD)
  E11        0.235  — Type 2 diabetes mellitus
  N18.3      0.176  — CKD, stage 3
  E11.9      0.176  — T2DM without complications
  I10        0.176  — Essential hypertension

  Betweenness Centrality (bridges between clinical domains):
  N18.3      0.007  — CKD, stage 3
  N18.6      0.004  — End stage renal disease
  E11.9      0.004  — T2DM without complications
  I50.32     0.004  — Chronic systolic HF
  N18        0.000  — Chronic kidney disease (CKD)

💡 High degree centrality = commonly co-coded diagnosis = high CDI query priority.
   I10 (hypertension) being central means: when you see hypertension, always
   check for co-coded CKD, diabetes, and heart failure — they drive DRG weight.

   Betweenness centrality = diagnostic bridges: N18.3 connects renal, endocrine,
   

### KG Visualisation Colour Legend

| Colour | Domain | Meaning |
|--------|--------|---------|
| 🟢 Teal `#0E7C7B` | `clinical` | Diagnosis codes used in CDS models |
| 🟡 Amber `#F59E0B` | `rcm` | Codes that drive reimbursement in RCM pipeline |
| 🔵 Navy `#0D2B45` | `engagement` | Conditions targeted in patient outreach |
| 🔴 Red `#DC2626` | `qualifier` | Documentation requirements (stage/type/laterality) |

**Edge types:** Grey = `PARENT_OF` (ICD-10 hierarchy) | Teal = `COMMONLY_CO_CODED` | Red dashed = `REQUIRES` qualifier

In [11]:
# ── PyVis interactive visualisation ───────────────────────────────────
try:
    from pyvis.network import Network
    from IPython.display import IFrame
    import os, tempfile

    DOMAIN_COLOURS = {
        'clinical':   '#0E7C7B',
        'rcm':        '#F59E0B',
        'engagement': '#0D2B45',
        'qualifier':  '#DC2626',
    }
    REL_COLOURS = {
        'PARENT_OF':          '#6B7280',
        'COMMONLY_CO_CODED':  '#0E7C7B',
        'REQUIRES':           '#DC2626',
    }

    net = Network(height='500px', width='100%', bgcolor='#F9FAFB', font_color='#374151',
                  directed=True, notebook=True, cdn_resources='in_line')

    for node, attrs in G.nodes(data=True):
        colour = DOMAIN_COLOURS.get(attrs.get('domain', 'clinical'), '#9CA3AF')
        label  = f"{node}\n{attrs.get('desc', '')[:30]}"
        net.add_node(node, label=label, color=colour, title=attrs.get('desc', node),
                     size=20 if '.' not in node else 14)

    for src, tgt, data in G.edges(data=True):
        rel    = data.get('rel', '')
        colour = REL_COLOURS.get(rel, '#9CA3AF')
        net.add_edge(src, tgt, title=rel, color=colour,
                     dashes=(rel == 'REQUIRES'))

    net.set_options('{"physics":{"stabilization":{"iterations":100}}}')
    out_path = os.path.join(tempfile.gettempdir(), 'cliniq_kg.html')
    # Generate HTML then write with explicit UTF-8 encoding (Windows fix)
    net.generate_html()
    with open(out_path, 'w', encoding='utf-8') as f:
        f.write(net.html)
    display(IFrame(out_path, width='100%', height='520px'))
    print('Knowledge graph rendered above')
except Exception as e:
    print(f'PyVis visualisation skipped: {e}')
    # Fallback: print graph summary
    print(f'\nKG Summary: {G.number_of_nodes()} nodes, {G.number_of_edges()} edges')
    for node in list(G.nodes())[:10]:
        desc = G.nodes[node].get('desc', node)
        print(f'  {node}: {desc}')

Knowledge graph rendered above


---
## 📋 Production Data Requirements

These architectures are prototypes. Before production deployment:

| Architecture | Minimum labelled data | Fine-tuning target | Evaluation metric |
|-------------|----------------------|-------------------|-------------------|
| **C-TFT** | 50k patient longitudinal sequences | Top-5 diagnosis recall | AUROC per condition |
| **HDE-CGD** | 200k coded encounters | ICD-10 top-3 code accuracy | MRR, code-valid rate |
| **MOCB-LLM** | 30-day outcome labels for 20k engagements | Cumulative reward vs oracle | Regret, equity DPD |

**KG embedding training:** In production, ICD-10 node2vec embeddings are trained on co-occurrence statistics from 3M+ real encounters. The graph structure + random-walk sampling gives ~128-dim embeddings that encode clinical similarity (N18.3 ↔ E11.9 are close; I10 ↔ N18.3 also close).

> **Design note:** GraphSAGE or TransE could be used for KG embeddings, trained on the EHR encounter co-occurrence matrix. The resulting embeddings would be frozen as patient-specific features injected into C-TFT and used as decoder priors in HDE-CGD.